# Train model and log with MLflow

In [1]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import os

/home/anuarsantoyo/Projects/Projects/MasterSchool/TimeSeriesProject_Jun26/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:


# Set the tracking URI to your custom folder
#mlflow.set_tracking_uri(f"file:///home/anuarsantoyo/PycharmProjects/masterschool/ml_flow_tut/mlruns")

experiment_name = "timeseries_project_forecasting"
mlflow.set_experiment(experiment_name)

2026/06/23 11:48:41 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/23 11:48:41 INFO mlflow.store.db.utils: Updating database tables
2026/06/23 11:48:43 INFO mlflow.tracking.fluent: Experiment with name 'timeseries_project_forecasting' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/anuarsantoyo/Projects/Projects/MasterSchool/TimeSeriesProject_Jun26/mlruns/1', creation_time=1782208123568, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1782208123568, lifecycle_stage='active', name='timeseries_project_forecasting', tags={}, trace_location=None, workspace='default'>

In [3]:

# Generate sample data
X, y = make_regression(n_samples=100, n_features=2, noise=0.1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create fit and predict with a model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)

# Calculate some metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)


In [4]:
    # Experiment 1: Linear Regression
with mlflow.start_run(run_name="Linear_Regression"):
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)
    mlflow.sklearn.log_model(lr_model, "model")

    print(f"Linear Regression - MSE: {mse:.4f}, R2: {r2:.4f}")

2026/06/23 11:49:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Linear Regression - MSE: 0.0155, R2: 1.0000


In [5]:

# Experiment 2: Ridge Regression

with mlflow.start_run(run_name="Ridge_Regression"):
    ridge_model = Ridge(alpha=1.0)
    ridge_model.fit(X_train, y_train)
    y_pred = ridge_model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    mlflow.log_param("model_type", "Ridge")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)
    mlflow.sklearn.log_model(ridge_model, "model") #  adapt this to your model!! IN this example I used an sklearn model, that is why I used mlfow.sklearn.log_model()


    print(f"Ridge Regression - MSE: {mse:.4f}, R2: {r2:.4f}")


2026/06/23 11:49:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Ridge Regression - MSE: 1.8170, R2: 0.9998


# Terminal commandos:
To open the MLflow UI, run the following command in your terminal:
mlflow ui

In case already in use error:
pkill -f mlflow

# Load and use a logged model

In [6]:
import mlflow
import mlflow.sklearn

# You need the run ID - get it from MLflow UI or by searching
run_id = "c7cdd6faec9e4ca0803306a0243adfb9"  # Replace with your actual run ID

# Load the model
model_uri = f"runs:/{run_id}/model"  # or "/linear_regression_model" if you used custom name
loaded_model = mlflow.sklearn.load_model(model_uri)

print("✅ Model loaded successfully!")
print(f"Model type: {type(loaded_model)}")

✅ Model loaded successfully!
Model type: <class 'sklearn.linear_model._base.LinearRegression'>


In [7]:
loaded_model.predict(X_test)

array([ -58.20320564,   -0.99412493,   84.42053424,  -19.80868937,
         67.78544042,  115.08677163,  195.3800306 , -126.83541741,
       -185.65676039,  -55.87134508,   80.26682995, -139.17457332,
        116.77784102,  -42.44949517,  112.75540523,   77.47491592,
        -27.1998263 ,    2.03152121,  -74.29575855,   43.73414793])

In [8]:

# Use the model for predictions
import numpy as np
sample_data = np.array([[1.0, 2.0]])
prediction = loaded_model.predict(sample_data)
print(f"Prediction: {prediction}")

Prediction: [235.86926742]
